Certainly! Below is a **comprehensive Python lab notebook-style guide**
that aligns with the content from your uploaded slides titled *“DQDC26 -
Module 4 - Standardization and Transformation.pdf”*.  
Each major concept (from slide 3 onward) is turned into a **step-by-step
code example**, using real or synthetic data where appropriate, with
explanations.

------------------------------------------------------------------------

# 🧪 **Lab: Data Standardization and Transformation Techniques**

> **Objective**: Implement and compare various data transformation
> techniques used in preprocessing for machine learning pipelines.

We’ll use: - `scikit-learn` - `pandas`, `numpy` - `matplotlib` /
`seaborn` for visualization - California Housing dataset (as referenced
in slides)

``` python
# Install if needed
# !pip install scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler,
    Normalizer, LabelEncoder, OrdinalEncoder, OneHotEncoder,
    PowerTransformer
)
```

------------------------------------------------------------------------

## 🔹 Step 1: Load & Inspect Data

``` python
# Load California Housing dataset
data = fetch_california_housing()
X, y = data.data, data.target
feature_names = data.feature_names

# Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print("Original data shape:", df.shape)
print("\nFirst few rows:")
df.head()
```

------------------------------------------------------------------------

## 🔹 Step 2: **Aesthetic Transformations** *(Slide 3)*

> These don’t change meaning—just format.

Since our dataset is numeric, we’ll simulate with a synthetic
categorical column:

``` python
# Simulate raw user input data
raw_df = pd.DataFrame({
    'address': ['123 Main St', '456 Oak street', '789 Pine ST'],
    'status': ['ACTIVE', 'inactive', 'Active'],
    'phone': ['(555) 123-4567', '555.987.6543', '5551234567']
})

# Aesthetic fixes
raw_df['address'] = raw_df['address'].str.replace(r'\bSt\b|\bstreet\b|\bST\b', 'Street', regex=True)
raw_df['status'] = raw_df['status'].str.capitalize()
raw_df['phone'] = raw_df['phone'].replace(r'\D', '', regex=True).apply(lambda x: f"({x[:3]}) {x[3:6]}-{x[6:10]}")

print("After aesthetic transformation:")
raw_df
```

------------------------------------------------------------------------

## 🔹 Step 3: **Constructive Transformations** *(Slide 3)*

Add derived features:

``` python
# Example: Add rooms per household and bedrooms ratio
df['rooms_per_household'] = df['AveRooms'] / df['Households']
df['bedroom_ratio'] = df['AveBedrms'] / df['AveRooms']

print("New features added:")
df[['rooms_per_household', 'bedroom_ratio']].head()
```

------------------------------------------------------------------------

## 🔹 Step 4: **Destructive Transformations** *(Slide 3)*

Remove duplicates & outliers:

``` python
# Remove duplicates (none expected here, but shown for completeness)
df_clean = df.drop_duplicates()

# Remove extreme outliers using IQR on target (median house value)
Q1 = df_clean['target'].quantile(0.25)
Q3 = df_clean['target'].quantile(0.75)
IQR = Q3 - Q1
outlier_mask = ~((df_clean['target'] < (Q1 - 1.5 * IQR)) | (df_clean['target'] > (Q3 + 1.5 * IQR)))
df_no_outliers = df_clean[outlier_mask]

print(f"Removed {len(df_clean) - len(df_no_outliers)} outlier rows.")
```

------------------------------------------------------------------------

## 🔹 Step 5: **Structural Transformations** *(Slide 3)*

Rename, reorder, convert types:

``` python
# Rename for clarity
df_renamed = df_no_outliers.rename(columns={'AveRooms': 'avg_rooms', 'MedInc': 'median_income'})

# Reorder columns: put target last
cols = [c for c in df_renamed.columns if c != 'target'] + ['target']
df_structured = df_renamed[cols]

# Ensure numeric types (already are, but good practice)
df_structured = df_structured.apply(pd.to_numeric, errors='coerce')

df_structured.head()
```

------------------------------------------------------------------------

## 🔹 Step 6: **Min-Max Scaling** *(Slides 5–7)*

Scale features to \[0, 1\]:

``` python
X = df_structured.drop('target', axis=1)
y = df_structured['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

minmax_scaler = MinMaxScaler()
X_train_minmax = minmax_scaler.fit_transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

print("Min values in test set after scaling:", X_test_minmax.min(axis=0).round(3))
print("Max values in test set after scaling:", X_test_minmax.max(axis=0).round(3))
```

------------------------------------------------------------------------

## 🔹 Step 7: **Standardization (Z-score)** *(Slides 8–10)*

Center at 0, std = 1:

``` python
std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train)
X_test_std = std_scaler.transform(X_test)

print("Mean after standardization (train):", X_train_std.mean(axis=0).round(3))
print("Std after standardization (train):", X_train_std.std(axis=0).round(3))
```

------------------------------------------------------------------------

## 🔹 Step 8: **L2 Normalization** *(Slides 11–12)*

Normalize each sample (row) to unit norm:

``` python
l2_normalizer = Normalizer(norm='l2')
X_train_l2 = l2_normalizer.fit_transform(X_train)
X_test_l2 = l2_normalizer.transform(X_test)

# Check norm of first row
print("L2 norm of first sample:", np.linalg.norm(X_train_l2[0]))
```

------------------------------------------------------------------------

## 🔹 Step 9: **Max-Abs Scaling** *(Slide 13)*

Scale by max absolute value → \[-1, 1\]:

``` python
maxabs_scaler = MaxAbsScaler()
X_train_maxabs = maxabs_scaler.fit_transform(X_train)
X_test_maxabs = maxabs_scaler.transform(X_test)

print("Max abs scaled range (approx):", X_test_maxabs.min(), "to", X_test_maxabs.max())
```

------------------------------------------------------------------------

## 🔹 Step 10: **Robust Scaling (IQR-based)** *(Slide 13)*

Robust to outliers:

``` python
robust_scaler = RobustScaler()
X_train_robust = robust_scaler.fit_transform(X_train)
X_test_robust = robust_scaler.transform(X_test)

print("Median (should be ~0):", np.median(X_train_robust, axis=0)[:3])
```

------------------------------------------------------------------------

## 🔹 Step 11: **Categorical Encoding Examples** *(Slides 14–19)*

We’ll create synthetic categorical

``` python
cat_data = pd.DataFrame({
    'color': ['Red', 'Green', 'Blue', 'Green', 'Red'],
    'size': ['S', 'XS', 'L', 'M', 'S'],
    'city': ['London', 'Manchester', 'Bristol', 'London', 'Bristol'],
    'rating': ['Poor', 'Good', 'Excellent', 'Good', 'Poor']
})
```

### ✅ One-Hot Encoding (Nominal)

``` python
ohe = OneHotEncoder(sparse_output=False)
encoded_ohe = ohe.fit_transform(cat_data[['color', 'city']])
pd.DataFrame(encoded_ohe, columns=ohe.get_feature_names_out())
```

### ✅ Label Encoding (Ordinal or Tree Models)

``` python
le = LabelEncoder()
cat_data['color_encoded'] = le.fit_transform(cat_data['color'])
cat_data[['color', 'color_encoded']]
```

### ✅ Ordinal Encoding (with order)

``` python
size_order = ['XS', 'S', 'M', 'L']
ord_enc = OrdinalEncoder(categories=[size_order])
cat_data['size_ord'] = ord_enc.fit_transform(cat_data[['size']])
cat_data[['size', 'size_ord']]
```

### ✅ Target Encoding (simulate with target)

``` python
# Simulated target
cat_data['price'] = [100, 150, 200, 160, 110]

# Target encode 'city'
target_mean = cat_data.groupby('city')['price'].mean()
cat_data['city_target_enc'] = cat_data['city'].map(target_mean)
cat_data
```

Continuing from where we left off, here is the **complete lab** covering
**numerical variable transformations** (Slides 20–26) and wrapping up
with best practices.

------------------------------------------------------------------------

## 🔹 Step 12: **Why Transform Numerical Variables?** *(Slide 20)*

Many ML models (e.g., linear regression, SVM) assume: - Linear
relationships - Normally distributed errors - Homoscedasticity (constant
variance)

Skewed features violate these assumptions. We’ll visualize and fix skew.

``` python
# Check skewness of original features
skew_vals = df_structured.drop('target', axis=1).skew().sort_values(ascending=False)
print("Top skewed features:")
skew_vals.head()
```

------------------------------------------------------------------------

## 🔹 Step 13: **Q-Q Plot to Assess Normality** *(Slide 21)*

``` python
import scipy.stats as stats

plt.figure(figsize=(8, 4))
stats.probplot(df_structured['median_income'], dist="norm", plot=plt)
plt.title("Q-Q Plot for Median Income")
plt.show()
```

> If points deviate from the diagonal → non-normal → consider
> transformation.

------------------------------------------------------------------------

## 🔹 Step 14: **Log Transformation** *(Slide 22)*

Useful for right-skewed

``` python
from sklearn.preprocessing import FunctionTransformer

# Ensure no zeros/negatives
X_log_safe = X_train.copy()
X_log_safe = np.where(X_log_safe <= 0, 1e-6, X_log_safe)  # Avoid log(0)

log_transformer = FunctionTransformer(np.log1p)  # log(x + 1)
X_train_log = log_transformer.fit_transform(X_log_safe)
X_test_log = log_transformer.transform(np.where(X_test <= 0, 1e-6, X_test))

# Compare skew before/after
orig_skew = pd.Series(X_train[:, feature_names.index('MedInc')]).skew()
new_skew = pd.Series(X_train_log[:, feature_names.index('MedInc')]).skew()
print(f"Original skew: {orig_skew:.2f} → Log-transformed skew: {new_skew:.2f}")
```

------------------------------------------------------------------------

## 🔹 Step 15: **Reciprocal & Square Root** *(Slide 23)*

``` python
# Reciprocal (for strong right skew)
recip_transformer = FunctionTransformer(lambda x: 1 / (x + 1e-6))
X_train_recip = recip_transformer.fit_transform(X_train)

# Square root (for count-like data)
sqrt_transformer = FunctionTransformer(np.sqrt)
X_train_sqrt = sqrt_transformer.fit_transform(np.abs(X_train))  # abs for safety
```

------------------------------------------------------------------------

## 🔹 Step 16: **Box-Cox Transformation** *(Slides 24–25)*

Only works for **positive values**:

``` python
from sklearn.preprocessing import PowerTransformer

# Box-Cox requires strictly positive data
X_pos = X_train[X_train > 0].min()  # ensure positivity
X_train_bc = X_train.copy()
X_train_bc = np.clip(X_train_bc, 1e-6, None)  # avoid zero/negative

boxcox = PowerTransformer(method='box-cox', standardize=True)
X_train_boxcox = boxcox.fit_transform(X_train_bc)
X_test_boxcox = boxcox.transform(np.clip(X_test, 1e-6, None))

# Verify reduced skew
bc_skew = pd.Series(X_train_boxcox[:, 0]).skew()
print(f"Box-Cox skew (first feature): {bc_skew:.2f}")
```

------------------------------------------------------------------------

## 🔹 Step 17: **Yeo-Johnson Transformation** *(Slide 26)*

Works with **zero, negative, and positive** values:

``` python
yeo = PowerTransformer(method='yeo-johnson', standardize=True)
X_train_yj = yeo.fit_transform(X_train)
X_test_yj = yeo.transform(X_test)

# Compare distribution
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(X_train[:, 0], bins=30, alpha=0.7, label='Original')
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(X_train_yj[:, 0], bins=30, alpha=0.7, color='orange', label='Yeo-Johnson')
plt.legend()
plt.suptitle("Effect of Yeo-Johnson on First Feature")
plt.show()
```

------------------------------------------------------------------------

## ✅ **Summary Table of Techniques**

| Technique        | Use Case                            | Handles Negatives? | Bounded?    | Outlier Robust?   |
|---------------|----------------------------|-------------|-------|-----------|
| Min-Max Scaling  | Neural nets, image data             | Yes                | Yes \[0,1\] | ❌                |
| Standardization  | Linear models, PCA                  | Yes                | No          | ❌                |
| Robust Scaling   | Data with outliers                  | Yes                | No          | ✅                |
| MaxAbs Scaling   | Sparse data                         | Yes                | \[-1,1\]    | ❌                |
| L2 Normalization | Text, clustering, cosine similarity | Yes                | Unit norm   | ❌                |
| One-Hot Encoding | Nominal low-cardinality             | —                  | —           | —                 |
| Target Encoding  | High-cardinality categories         | —                  | —           | ⚠️ (leakage risk) |
| Log / Box-Cox    | Right-skewed numeric                | Box-Cox: ❌        | No          | ❌                |
| Yeo-Johnson      | Skewed numeric (any sign)           | ✅                 | No          | ❌                |

------------------------------------------------------------------------

## 🧪 Final Lab Exercise (Optional)

**Task**: Build a simple pipeline comparing model performance (e.g.,
Ridge Regression) using: - Raw data - MinMax-scaled data - Yeo-Johnson +
Standardized data

Which gives lowest RMSE?

``` python
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

models = {
    'Raw': X_train,
    'MinMax': X_train_minmax,
    'YeoJohnson+Std': X_train_yj  # already standardized by PowerTransformer
}

results = {}
for name, X in models.items():
    model = Ridge(alpha=1.0).fit(X, y_train)
    y_pred = model.predict(models[name] if name != 'Raw' else X_train)  # train eval
    rmse = np.sqrt(mean_squared_error(y_train, y_pred))
    results[name] = rmse

print("Train RMSE comparison:", results)
```

> 💡 **Tip**: Always transform **only on training data**, then apply
> same transformer to test set!

------------------------------------------------------------------------

Let me know if you’d like this as a downloadable `.ipynb` file or want
to add visual comparisons (e.g., histograms before/after each
transform)!